### Simulation example using the NeuroWorkflow library.

This example demonstrates how to create a basic workflow for Virtual Neuromodulation (Group Surrogate model)
using the NeuroWorkflow library.

In [1]:
#!pip install hdf5storage

In [2]:
#!pip install itk itkwidgets

In [3]:
#!pip install ipyniivue

In [1]:
import sys
import os

# Add the src directory to the Python path if needed
src_path = os.path.abspath(os.path.join(os.getcwd(), '../src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    

from neuroworkflow import WorkflowBuilder
from neuroworkflow.nodes.io.VNMLoadSubjectTimeSeriesNode import VNMLoadSubjectTimeSeriesNode
from neuroworkflow.nodes.io.VNMLoadGroupSurrogateModelNode import VNMLoadGroupSurrogateModelNode
from neuroworkflow.nodes.simulation.VNMSimulatorNode import VNMSimulatorNode
from neuroworkflow.nodes.analysis.VNMGlmAnalysisNode import VNMGlmAnalysisNode
from neuroworkflow.nodes.analysis.NiftiViewerNode import NiftiViewerNode
from neuroworkflow.nodes.analysis.NiftiViewer3DNode import NiftiViewer3DNode


In [2]:
# Monkey-patch: replace process-based GLM with thread-based (macOS safe)
#only for mac, remove or comment this cell for ubuntu 
import neuroworkflow.utils.vneumodpy as vnm
from neuroworkflow.utils.vneumodpy.glm.tukey_mth import calc as _tukey_mth

def tukey_mth_compat(Y, X, tuM=None, isOutX2is=False, n_jobs=None):
    return _tukey_mth(Y, X, tuM=tuM, isOutX2is=isOutX2is)
vnm.tukey_mp = tukey_mth_compat

In [3]:
#!pip install --upgrade statsmodels (updated to 0.14.6 , with scipy: 1.16.2 )

In [4]:
# Create nodes
load_subject = VNMLoadSubjectTimeSeriesNode("VNMLoadSubject")
load_subject.configure(
    cx_file="../data/vnm_data/subjects/ppmi81CXAllenCube2s34gmacomp.mat",
    atlas_file="../data/vnm_data/atlas/allenCube2atlas.nii.gz",
)

load_model = VNMLoadGroupSurrogateModelNode("VNMLoadModel")
load_model.configure(
    model_file="../data/vnm_data/model/ppmi81CXAllenCube2s34gmacomp_gsm_var.mat",
)

simulate_vnm = VNMSimulatorNode("VNMSimulation")
simulate_vnm.configure(
    target_ROI_file="../data/vnm_data/target_roi/allenCube2atlasStn3.nii.gz",
    target_ROI=[4525],
    simulation_name="ppmi81CXAllenCube2s34gmacomp",
    subject_perm_path="../data/vnm_data/perm",
    number_of_trials=1
)

analysis_vnm = VNMGlmAnalysisNode("VNMGlmAnalysis")
analysis_vnm.configure(
    atlas_file="../data/vnm_data/atlas/allenCube2atlas.nii.gz",
    result_nifti_path="../data/vnm_data/output",
)

viewer = NiftiViewerNode("NiftiViewer")
viewer.configure(
    bg_img="../data/vnm_data/atlas/allenCube2atlas.nii.gz",
    cal_min=1.0,
    cal_max=8.0,
)

#viewer_3d = NiftiViewer3DNode("NiftiViewer3D")
#viewer_3d.configure(
    #bg_img="../data/vnm_data/atlas/allenCube2atlas.nii.gz",
#    cal_min=1.5,
#    cal_max=8.0,
#)


In [5]:
# Create workflow
workflow = (
    WorkflowBuilder("vnm_simulation_analysis")
        .add_node(load_subject)
        .add_node(load_model)
        .add_node(simulate_vnm)
        .add_node(analysis_vnm)
        .add_node(viewer)
        #.add_node(viewer_3d)
        .connect("VNMLoadSubject", "CX", "VNMSimulation", "CX")
        .connect("VNMLoadSubject", "atlasV", "VNMSimulation", "atlasV")
        .connect("VNMLoadModel", "model", "VNMSimulation", "model")
        .connect("VNMSimulation", "simulation_name", "VNMGlmAnalysis", "simulation_name")
        .connect("VNMSimulation", "trials", "VNMGlmAnalysis", "trials")
        .connect("VNMSimulation", "Chrf", "VNMGlmAnalysis", "Chrf")
        .connect("VNMGlmAnalysis", "nifti_files", "NiftiViewer", "nifti_files")
        #.connect("VNMGlmAnalysis", "nifti_files", "NiftiViewer3D", "nifti_files")
        .build()
    )

In [6]:
# Print workflow information
print(workflow)
    
# Execute workflow
print("\nExecuting workflow...")
success = workflow.execute()
    
if success:
    print("Workflow execution completed successfully!")
else:
    print("Workflow execution failed!")

Workflow: vnm_simulation_analysis
Nodes:
  VNMLoadSubject
  VNMLoadModel
  VNMSimulation
  VNMGlmAnalysis
  NiftiViewer
Connections:
  VNMLoadSubject.CX -> VNMSimulation.CX
  VNMLoadSubject.atlasV -> VNMSimulation.atlasV
  VNMLoadModel.model -> VNMSimulation.model
  VNMSimulation.simulation_name -> VNMGlmAnalysis.simulation_name
  VNMSimulation.trials -> VNMGlmAnalysis.trials
  VNMSimulation.Chrf -> VNMGlmAnalysis.Chrf
  VNMGlmAnalysis.nifti_files -> NiftiViewer.nifti_files

Executing workflow...
Executing node: VNMLoadModel
Loading Group Surrogate model : ../data/vnm_data/model/ppmi81CXAllenCube2s34gmacomp_gsm_var.mat
Executing node: VNMLoadSubject
Loading Subject multivariate time-series : ../data/vnm_data/subjects/ppmi81CXAllenCube2s34gmacomp.mat
Loading Cube ROI atlas : ../data/vnm_data/atlas/allenCube2atlas.nii.gz
Executing node: VNMSimulation
Loading target ROI nifti file: ../data/vnm_data/target_roi/allenCube2atlasStn3.nii.gz
generate modulation (add & mul) time-series, target r

Workflow execution completed successfully!


In [7]:
#!jupyter lab --version

In [11]:
#!pip show comm
#!pip show jupyterlab_widgets

In [12]:
#!pip show ipywidgets

In [13]:
#!pip install "ipywidgets>=8.0"